# Kirsch Visualizer — Near-Wellbore Stresses (QC)

Kirsch solution on the borehole wall via GeomechPy.

**Units:** all stresses in **psi** (not ksi).

Polar and Cartesian QC plots of radial, tangential, axial and principal stresses.


## Setup & imports


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "Project":
    REPO_ROOT = REPO_ROOT.parent.parent
elif REPO_ROOT.name == "example":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from geomechpy.near_wellbore_stresses import NearWellboreStressesCalculation

print("Imports OK — repo root:", REPO_ROOT)


## Input parameters (single depth) — psi

Edit values to explore stress regimes and trajectories.


In [ ]:
SHMIN = 6500.0
SHMAX = 8500.0
SVERT = 10000.0
PORE_PRESSURE = 4500.0
MUD_PRESSURE = 5000.0

SHMAX_AZIMUTH = 30.0
BOREHOLE_DEVIATION = 0.0
BOREHOLE_AZIMUTH = 0.0
POISSON_RATIO_STATIC = 0.25

theta = np.linspace(0, 360, 361)

print(f"Far-field (psi): Sv={SVERT}, SHmax={SHMAX}, Shmin={SHMIN}, Pp={PORE_PRESSURE}, Pw={MUD_PRESSURE}")


## Compute Kirsch borehole-wall stresses


In [ ]:
wall = NearWellboreStressesCalculation.calculate_kirsch_borehole_wall_stresses(
    shmin=SHMIN,
    shmax=SHMAX,
    svert=SVERT,
    pore_pressure=PORE_PRESSURE,
    shmax_azimuth=SHMAX_AZIMUTH,
    mud_pressure=MUD_PRESSURE,
    theta=theta,
    poisson_ratio_static=POISSON_RATIO_STATIC,
    borehole_deviation=BOREHOLE_DEVIATION,
    borehole_azimuth=BOREHOLE_AZIMUTH,
)

principals = NearWellboreStressesCalculation.calculate_principal_stresses_analytical(
    sigma_tt=wall.sigma_tt,
    sigma_zz=wall.sigma_zz,
    sigma_tz=wall.sigma_tz,
)

print("sigma_rr range (psi):", float(wall.sigma_rr.min()), "–", float(wall.sigma_rr.max()))
print("sigma_tt range (psi):", float(wall.sigma_tt.min()), "–", float(wall.sigma_tt.max()))
print("sigma_zz range (psi):", float(wall.sigma_zz.min()), "–", float(wall.sigma_zz.max()))
print("sigma_1  range (psi):", float(principals.sigma_1.min()), "–", float(principals.sigma_1.max()))
print("sigma_2  range (psi):", float(principals.sigma_2.min()), "–", float(principals.sigma_2.max()))


### QC plot — polar stresses (psi)


In [ ]:
theta_rad = np.deg2rad(theta)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), subplot_kw=dict(projection="polar"))

axes[0].plot(theta_rad, wall.sigma_tt, "b-", lw=2)
axes[0].fill_between(theta_rad, 0, wall.sigma_tt, alpha=0.25, color="b")
axes[0].set_title(r"$\sigma_{\theta\theta}$ (psi)", pad=20)
axes[0].set_theta_zero_location("N"); axes[0].set_theta_direction(-1)

axes[1].plot(theta_rad, wall.sigma_zz, "g-", lw=2)
axes[1].fill_between(theta_rad, 0, wall.sigma_zz, alpha=0.25, color="g")
axes[1].set_title(r"$\sigma_{zz}$ (psi)", pad=20)
axes[1].set_theta_zero_location("N"); axes[1].set_theta_direction(-1)

axes[2].plot(theta_rad, principals.sigma_1, "r-", lw=2)
axes[2].fill_between(theta_rad, 0, principals.sigma_1, alpha=0.25, color="r")
axes[2].set_title(r"$\sigma_1$ principal (psi)", pad=20)
axes[2].set_theta_zero_location("N"); axes[2].set_theta_direction(-1)

plt.suptitle(
    f"QC-1 Kirsch polar (psi) | SHmax az={SHMAX_AZIMUTH}° | "
    f"dev={BOREHOLE_DEVIATION}° | Pw={MUD_PRESSURE:.0f} psi",
    fontsize=11,
)
plt.tight_layout(); plt.show()


### QC plot — Cartesian stresses vs azimuth (psi)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(theta, wall.sigma_tt, "b-", label=r"$\sigma_{\theta\theta}$")
ax.plot(theta, wall.sigma_zz, "g-", label=r"$\sigma_{zz}$")
ax.plot(theta, wall.sigma_rr, "k--", label=r"$\sigma_{rr}$")
ax.plot(theta, principals.sigma_1, "r-", lw=2, label=r"$\sigma_1$")
ax.plot(theta, principals.sigma_2, "m-", lw=2, label=r"$\sigma_2$")
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel(r"$\theta$ (deg from Top-of-Hole)")
ax.set_ylabel("Stress (psi)")
ax.set_title("QC-2 Borehole-wall stresses vs azimuth (psi)")
ax.legend(); ax.grid(True, alpha=0.3); ax.set_xlim(0, 360)
plt.tight_layout(); plt.show()


### QC plot — all wall components including shear (psi)

Full component set for equation QC: $\sigma_{rr}$, $\sigma_{\theta\theta}$, $\sigma_{zz}$, $\sigma_{\theta z}$.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)

axes[0, 0].plot(theta, wall.sigma_rr, "k-"); axes[0, 0].set_ylabel("psi"); axes[0, 0].set_title(r"$\sigma_{rr}$")
axes[0, 1].plot(theta, wall.sigma_tt, "b-"); axes[0, 1].set_title(r"$\sigma_{\theta\theta}$")
axes[1, 0].plot(theta, wall.sigma_zz, "g-"); axes[1, 0].set_xlabel(r"$\theta$ (deg)"); axes[1, 0].set_ylabel("psi"); axes[1, 0].set_title(r"$\sigma_{zz}$")
axes[1, 1].plot(theta, wall.sigma_tz, "m-"); axes[1, 1].set_xlabel(r"$\theta$ (deg)"); axes[1, 1].set_title(r"$\sigma_{\theta z}$")

for ax in axes.ravel():
    ax.grid(True, alpha=0.3); ax.set_xlim(0, 360)
plt.suptitle("QC-3 Full Kirsch wall components (psi)", fontsize=12)
plt.tight_layout(); plt.show()


### QC plot — vertical vs horizontal well (psi)


In [ ]:
def run_kirsch(deviation):
    w = NearWellboreStressesCalculation.calculate_kirsch_borehole_wall_stresses(
        shmin=SHMIN, shmax=SHMAX, svert=SVERT,
        pore_pressure=PORE_PRESSURE, shmax_azimuth=SHMAX_AZIMUTH,
        mud_pressure=MUD_PRESSURE, theta=theta,
        poisson_ratio_static=POISSON_RATIO_STATIC,
        borehole_deviation=deviation, borehole_azimuth=BOREHOLE_AZIMUTH,
    )
    p = NearWellboreStressesCalculation.calculate_principal_stresses_analytical(
        sigma_tt=w.sigma_tt, sigma_zz=w.sigma_zz, sigma_tz=w.sigma_tz,
    )
    return w, p

wall_v, prin_v = run_kirsch(0.0)
wall_h, prin_h = run_kirsch(90.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), subplot_kw=dict(projection="polar"))

axes[0].plot(theta_rad, prin_v.sigma_1, "r-", lw=2, label=r"$\sigma_1$")
axes[0].plot(theta_rad, prin_v.sigma_2, "b-", lw=2, label=r"$\sigma_2$")
axes[0].set_title("Vertical well (dev=0°)", pad=20)
axes[0].set_theta_zero_location("N"); axes[0].set_theta_direction(-1)
axes[0].legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))

axes[1].plot(theta_rad, prin_h.sigma_1, "r-", lw=2, label=r"$\sigma_1$")
axes[1].plot(theta_rad, prin_h.sigma_2, "b-", lw=2, label=r"$\sigma_2$")
axes[1].set_title("Horizontal well (dev=90°)", pad=20)
axes[1].set_theta_zero_location("N"); axes[1].set_theta_direction(-1)

plt.suptitle("QC-4 Principal wall stresses (psi) — vertical vs horizontal", fontsize=12)
plt.tight_layout(); plt.show()


## Notes

- All stress axes are in **psi**.
- $\theta = 0°$ is Top-of-Hole (TOH).
- Breakouts form where $\sigma_{\theta\theta}$ is maximum (typically along Shmin).
- Tensile fractures initiate where $\sigma_{\theta\theta}$ is minimum (along SHmax).
- Change deviation, SHmax azimuth or mud pressure and re-run for QC.
